# Lecture 3 — Class Exercise
## Line Charts & Slopegraphs: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
1. No spaghetti — multiple lines must use grey + single highlight
2. Remove clutter: no chart borders, no heavy gridlines, no legend if you can label directly
3. Insight title — states the finding, not the topic
4. Carry forward from Lecture 2: white background, Arial font, professional quality


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('/content/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [2]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


---
## Task 1 — Multi-Series Line Chart with Highlight

**What to build:** A line chart showing CO2 emissions over time for **all Asian countries** in the dataset, with one country highlighted.

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#DDDDDD), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

> 💡 `df[df['Region'] == 'Asia']` to filter; use `go.Figure()` with a loop for per-country control


In [3]:
# Task 1 — Multi-series line with highlight
# -------------------------------------------
asia = df[df['Region'] == 'Asia']

highlight = 'China'
grey = '#DDDDDD'
highlight_color = '#C0392B'

fig1 = go.Figure()

# Grey context lines for all non-highlighted countries
for country in asia['Country'].unique():
    if country == highlight:
        continue
    sub = asia[asia['Country'] == country].sort_values('Year')
    fig1.add_trace(go.Scatter(
        x=sub['Year'], y=sub['CO2_Mt'],
        mode='lines',
        line=dict(color=grey, width=1.5),
        showlegend=False,
        hoverinfo='skip',
    ))

# Highlighted line on top
sub = asia[asia['Country'] == highlight].sort_values('Year')
fig1.add_trace(go.Scatter(
    x=sub['Year'], y=sub['CO2_Mt'],
    mode='lines',
    line=dict(color=highlight_color, width=3),
    showlegend=False,
    hoverinfo='skip',
))

# Direct end-of-line label instead of a legend
last_year = sub['Year'].max()
last_val = sub[sub['Year'] == last_year]['CO2_Mt'].values[0]
fig1.add_annotation(
    x=last_year, y=last_val,
    text=f'<b>{highlight}</b>',
    showarrow=False,
    xanchor='left', xshift=10,
    font=dict(color=highlight_color, size=14),
)

start_val = sub[sub['Year'] == sub['Year'].min()]['CO2_Mt'].values[0]
growth_x = last_val / start_val

fig1.update_layout(
    title=dict(
        text=(f"<b>China's CO2 emissions grew {growth_x:.1f}x since 2000, far outpacing the rest of Asia</b>"
              f"<br><sup>Annual CO2 emissions by country, Asia, {asia['Year'].min()}-{asia['Year'].max()}</sup>"),
        x=0, xanchor='left', font=dict(size=16)
    ),
    xaxis=dict(title='', showgrid=False, showline=True, linecolor='#cccccc'),
    yaxis=dict(title='CO2 Emissions (Mt)', showgrid=True, gridcolor='#eeeeee', zeroline=False),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial, sans-serif', size=13, color='#333'),
    height=500,
    width=850,
    margin=dict(l=10, r=90, t=90, b=40),
    showlegend=False,
)

fig1.show()


---
## Task 2 — Slopegraph: Regional Change 2000 vs 2022

**What to build:** A slopegraph comparing **average regional CO2 emissions** between 2000 and 2022.

**Requirements:**
- One line per region (not per country — aggregate first)
- Colour: regions that increased = one colour; decreased = another
- Values labelled at both ends of each line
- No y-axis tick labels (the endpoint labels make them redundant)
- Insight title stating which regions moved most

> 💡 `df.groupby(['Region','Year'])['CO2_Mt'].mean().reset_index()` then filter to 2000 and 2022


In [4]:
# Task 2 — Slopegraph: regional averages
# -----------------------------------------
reg_avg = df.groupby(['Region', 'Year'])['CO2_Mt'].mean().reset_index()
reg_2000 = reg_avg[reg_avg['Year'] == 2000].set_index('Region')['CO2_Mt']
reg_2022 = reg_avg[reg_avg['Year'] == 2022].set_index('Region')['CO2_Mt']

slope_df = pd.DataFrame({'2000': reg_2000, '2022': reg_2022})
slope_df['change'] = slope_df['2022'] - slope_df['2000']
slope_df = slope_df.sort_values('change')

increase_color = '#C0392B'
decrease_color = '#2E86AB'

fig2 = go.Figure()

for region, row in slope_df.iterrows():
    color = increase_color if row['change'] > 0 else decrease_color
    fig2.add_trace(go.Scatter(
        x=['2000', '2022'], y=[row['2000'], row['2022']],
        mode='lines+markers',
        line=dict(color=color, width=2.5),
        marker=dict(color=color, size=7),
        showlegend=False,
        hoverinfo='skip',
    ))
    # Labels at both ends: region name + value on the left, value only on the right
    fig2.add_annotation(x='2000', y=row['2000'], text=f"<b>{region}</b>  {row['2000']:.0f}",
                         showarrow=False, xanchor='right', xshift=-10,
                         font=dict(size=12, color=color))
    fig2.add_annotation(x='2022', y=row['2022'], text=f"{row['2022']:.0f}",
                         showarrow=False, xanchor='left', xshift=10,
                         font=dict(size=12, color=color))

biggest_mover = slope_df['change'].abs().idxmax()
biggest_change = slope_df.loc[biggest_mover, 'change']
direction = 'surged' if biggest_change > 0 else 'fell'

fig2.update_layout(
    title=dict(
        text=(f"<b>{biggest_mover}'s emissions {direction} the most, "
              f"{'up' if biggest_change > 0 else 'down'} {abs(biggest_change):.0f} Mt since 2000</b>"
              f"<br><sup>Average CO2 emissions per country, by region: 2000 vs 2022</sup>"),
        x=0, xanchor='left', font=dict(size=16)
    ),
    xaxis=dict(title='', showgrid=False, showline=False,
               tickfont=dict(size=13)),
    yaxis=dict(title='', showticklabels=False, showgrid=False, zeroline=False),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial, sans-serif', size=13, color='#333'),
    height=550,
    width=750,
    margin=dict(l=160, r=90, t=100, b=40),
    showlegend=False,
)

fig2.show()
